In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import importlib
from config import *
from simulation.aggregate_metrics import aggregate_metrics


In [2]:
import pandas as pd
from pathlib import Path

# Folder containing the Excel files
folder = Path("../results")

# Find all Excel files starting with "results_"
files = sorted(folder.glob("results_*.xlsx"))

# Read and concatenate
metrics = pd.concat(
    (pd.read_excel(file, sheet_name="metrics") for file in files),
    ignore_index=True
)

print(f"Loaded {len(files)} files.")
print(metrics.head())

Loaded 108 files.
  method hv_selection       selection_type   bound_estimator  sample_size  \
0    MUS      nothing  systematic_sampling  Poisson_Stringer           30   
1    MUS      nothing  systematic_sampling  Poisson_Stringer           30   
2    MUS      nothing  systematic_sampling  Poisson_Stringer           30   
3    MUS      nothing  systematic_sampling  Poisson_Stringer           65   
4    MUS      nothing  systematic_sampling  Poisson_Stringer           65   

   confidence_level   z_score                        Population ID  \
0              0.80  0.841621  BV_15pct_above_SI_F0.05_C0.1_R0.002   
1              0.90  1.281552  BV_15pct_above_SI_F0.05_C0.1_R0.002   
2              0.95  1.644854  BV_15pct_above_SI_F0.05_C0.1_R0.002   
3              0.80  0.841621  BV_15pct_above_SI_F0.05_C0.1_R0.002   
4              0.90  1.281552  BV_15pct_above_SI_F0.05_C0.1_R0.002   

   Population Book Value  Population Error Amount  ...  \
0           3.425979e+09             6.8

In [3]:
import re

# Population ID format (see main.py's _population_id()): BVBV_<N>pct_above_SI_F<f>_C<c>_R<r>
# e.g. "BVBV_5pct_above_SI_F0.2_C0.1_R0.01" -> BV_pop="5pct", f_target=0.2, corr_target=0.1, r_target=0.01
pattern = r"^BV_(?P<bv>\d+pct)_above_SI_F(?P<f>[\d.]+)_C(?P<c>[\d.]+)_R(?P<r>[\d.]+)$"
extracted = metrics["Population ID"].astype(str).str.extract(pattern)

metrics["BV_pop"] = extracted["bv"]
metrics["f_target"] = extracted["f"].astype(float)
metrics["corr_target"] = extracted["c"].astype(float)
metrics["r_target"] = extracted["r"].astype(float)

In [4]:
path = RESULTS_DIR / "main_simulation_results.xlsx"
with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        metrics.to_excel(writer, sheet_name="metrics", index=False)
        for group_col, table in aggregate_metrics(metrics, "main").items():
            table.to_excel(writer, sheet_name=f"agg_{group_col}"[:31], index=True)